# 01 从原始数据到论文估计量

对应当前已接受的Human Reproduction v6；Overleaf r1仅修复编译。顺序：核对7输入 → Python完整计算 → 看清权重与参照 → 手算三项分解 → 亚型加和 → 02逐图 → 03独立R。

19国（中国+18参照），1990–2023，女性15–49岁七年龄组，两病因、三亚型。复现GBD下游描述性点估计分析，不重拟合IHME模型。无合格联合draws，不生成后验区间。


In [ ]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
ROOT = next(p for p in [here, *here.parents] if (p / "docs/20261030_INPUT_CHECKSUMS.csv").exists())
sys.path.insert(0, str(ROOT / "code"))
import reproduce as rep
import numpy as np
import pandas as pd
print("复现包：", ROOT.name)


## 1 核对输入
4份真实GBD CSV、UN人口子集、权重及UN来源清单。缺失或SHA256改变就停止。私人完整版已带齐；公开版须合法恢复GBD文件。

In [ ]:
rep.verify_inputs()

## 2 从原始CSV完整复算
原稿和输入只读；计算写入本包results/python、tables/python、figures/python。旧汇总只在计算完成后作回归核对。此Python步骤无需先安装R；独立R核对在03。

In [ ]:
rep.run_python()

## 3 标准化率与参照

由匹配GBD病例数/人口重建 $r_{cat}=10^5N_{cat}/P_{cat}$。七年龄权重先归一化 $w_a=w_a^*/\sum_aw_a^*$。

中国 $A_{CHN,t}=\sum_aw_ar_{CHN,at}$。参照国年龄内份额 $s_{cat}=P_{cat}/\sum_{c\in\mathcal R}P_{cat}$，合并参照率 $A_{\mathcal R,t}=\sum_aw_a\sum_cs_{cat}r_{cat}$。

$D_t=A_{CHN,t}-A_{\mathcal R,t}$。UN只替换人口份额，病率仍来自GBD。下面只显示权重与全国/合并参照汇总，不打印受限国家-年龄源单元。


In [ ]:
weights = pd.read_csv(ROOT / "data/input/weights.csv")
weights["normalized_weight"] = weights["weight"] / weights["weight"].sum()
display(weights)
a = pd.read_csv(ROOT / "results/python/all_annual_benchmark_aspr.csv")
display(a[(a.cause=="Polycystic ovarian syndrome") & (a.subtype=="Total") & (a.benchmark=="dynamic") & a.year.isin([1990,2023])])


## 4 从底层数组手算1990–2023三项分解

$$\Delta D=X+Q+S,$$
$$X=\sum_aw_a(r_{CHN,a1}-r_{CHN,a0}),$$
$$Q=-\sum_aw_a\sum_c\frac{s_{ca0}+s_{ca1}}2(r_{ca1}-r_{ca0}),$$
$$S=-\sum_aw_a\sum_c\frac{r_{ca0}+r_{ca1}}2(s_{ca1}-s_{ca0}).$$

三项都是带符号的绝对贡献。$100S/\Delta D$不是患病归因分数、后验概率或置信度，可以超出0–100%。计算下方数组后与完整计算结果核对。


In [ ]:
cells = pd.read_csv(ROOT / "results/python/rate_cells.csv")  # 受限单元只在私人包，勿公开
ages, years = list(range(15,50,5)), list(range(1990,2024))
refs = sorted(set(cells.country)-{"China"})
z = cells[(cells.cause=="Polycystic ovarian syndrome") & (cells.subtype=="Total")]
idx = pd.MultiIndex.from_product([refs,ages,years],names=["country","age_start","year"])
v = z.set_index(["country","age_start","year"]).reindex(idx)
r = v.gbd_rate.to_numpy().reshape(18,7,34)
p = v.gbd_pop.to_numpy().reshape(18,7,34)
s = p / p.sum(axis=0,keepdims=True)
cn = z[z.country=="China"].set_index(["age_start","year"]).reindex(pd.MultiIndex.from_product([ages,years])).gbd_rate.to_numpy().reshape(7,34)
w = weights.normalized_weight.to_numpy()
X = np.sum(w*(cn[:,-1]-cn[:,0]))
Q = -np.sum(w*np.sum((s[:,:,0]+s[:,:,-1])/2*(r[:,:,-1]-r[:,:,0]),axis=0))
S = -np.sum(w*np.sum((r[:,:,0]+r[:,:,-1])/2*(s[:,:,-1]-s[:,:,0]),axis=0))
net = np.sum(w*(cn[:,-1]-np.sum(s[:,:,-1]*r[:,:,-1],axis=0))) - np.sum(w*(cn[:,0]-np.sum(s[:,:,0]*r[:,:,0],axis=0)))
assert abs(X+Q+S-net)<1e-8
d = pd.read_csv(ROOT / "results/python/symmetric_decomposition_1990_endpoints.csv")
row = d[(d.cause=="Polycystic ovarian syndrome") & (d.subtype=="Total") & (d.source=="GBD") & (d.end==2023)].iloc[0]
assert np.max(abs(np.array([X,Q,S,net])-row[["china_rate_effect","reference_rate_effect","reference_country_composition_effect","net_gap_change"]].to_numpy(float)))<1e-8
display(pd.DataFrame({"component":["X","Q","S","net","100S/net"],"value":[X,Q,S,net,100*S/net]}))


## 5 亚型加和与v6补充S10
408行年度表逐行Total=Primary+Secondary。终点接近零可能来自原发/继发抵消，不证明历年稳定相同。

In [ ]:
annual_types = pd.read_csv(ROOT / "results/python/v6_annual_subtype_differences.csv")
assert len(annual_types)==408 and annual_types.sum_residual.abs().max()<1e-8
print("408行加和最大残差",annual_types.sum_residual.abs().max())
display(pd.read_csv(ROOT / "tables/python/TableS10.csv"))


## 6 下一步
到02选择recomputed逐图运行，最后03独立R验收。GATHER清单、S9文献比较及S11定义比较是来源/方法表；随包保存，不伪装成统计输出。